In [31]:
import scipy.io

# Load the .mat file into a dictionary
data = scipy.io.loadmat('/Users/Sam/dc-allen-data-acquire/data/mishler_results/LFP Data.mat')

# View all variable keys stored inside the file
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'LFP'])


In [32]:
# Extract a specific variable (replace 'variable_name' with your actual variable)
lfp = data['LFP']

In [33]:
lfp

array([[array([[ 4.3581740e-06, -3.3721095e-05, -6.9234826e-05, ...,
                 1.1421829e-05,  1.5667896e-05,  2.2821283e-05]],
              shape=(1, 2442112), dtype=float32)                    ,
        array([[-1.4536676e-04, -1.0097608e-04, -9.2629351e-05, ...,
                 7.8244848e-05,  6.4726839e-05,  5.0244344e-05]],
              shape=(1, 2442752), dtype=float32)                    ,
        array([[-5.9903937e-06,  2.1361555e-05,  3.1412113e-05, ...,
                -8.6765423e-05, -7.0890637e-05, -7.0031907e-05]],
              shape=(1, 2472448), dtype=float32)                    ],
       [array([[1.9483572e-05, 5.7597754e-06, 4.2656451e-05, ..., 9.8471130e-05,
                7.3143245e-05, 5.0212828e-05]], shape=(1, 2443520), dtype=float32),
        array([[ 1.13423186e-04,  3.78020268e-05,  4.67883319e-06, ...,
                -4.18359668e-05,  1.38778605e-05,  1.68536608e-05]],
              shape=(1, 2442240), dtype=float32)                       ,
     

In [34]:
# Load the .mat file into a dictionary
fGLFP_data = scipy.io.loadmat('/Users/Sam/dc-allen-data-acquire/data/mishler_results/finalGlutLFPMotifs.mat')

# View all variable keys stored inside the file
print(fGLFP_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'finalMotifs', 'shifts', 'finalMotifsAll'])


In [35]:
finalMotifs = fGLFP_data['finalMotifs']
shifts = fGLFP_data['shifts']
finalMotifsAll = fGLFP_data['finalMotifsAll']



In [8]:
import numpy as np
import scipy.io as sio

# --- 1. Inspect Table Headers and Field Names ---
def inspect_matlab_object(var, name="Variable"):
    print(f"=== {name} ===")
    print(f"Type: {type(var)}, Shape: {getattr(var, 'shape', None)}, Dtype: {getattr(var, 'dtype', None)}")
    if hasattr(var, 'dtype') and var.dtype.names:
        print(f"Field names: {var.dtype.names}\n")
    else:
        print(f"Sample element type: {type(var.flat[0]) if hasattr(var, 'flat') else 'N/A'}\n")

# Run inspections
inspect_matlab_object(shifts, "shifts")
inspect_matlab_object(finalMotifs, "finalMotifs")
inspect_matlab_object(finalMotifsAll, "finalMotifsAll")

=== shifts ===
Type: <class 'numpy.ndarray'>, Shape: (76, 21), Dtype: int32
Sample element type: <class 'numpy.int32'>

=== finalMotifs ===
Type: <class 'numpy.ndarray'>, Shape: (76, 6), Dtype: float64
Sample element type: <class 'numpy.float64'>

=== finalMotifsAll ===
Type: <class 'numpy.ndarray'>, Shape: (21, 5), Dtype: object
Sample element type: <class 'numpy.ndarray'>



In [34]:
import numpy as np

# Pick channel/session index 0, state index 0 (e.g. awake)
raw_entry = lfp[0, 0]

# Unwrap any Scipy 0D/1D object array wrappers
while isinstance(raw_entry, np.ndarray) and raw_entry.dtype == object:
    raw_entry = raw_entry[0]

# Ensure 1D float64 array
raw_1d = np.squeeze(raw_entry).astype(np.float64)

# Define sampling rate (e.g., 250 Hz, or 1000 Hz if undecimated)
# Check length: ~240,000 pts @ 1000 Hz = 240s; @ 250 Hz = 960s
fs = 250.0  # Or whatever sampling rate corresponds to this .mat file

# Slice a 60-second excerpt (60 * fs samples)
n_samples = int(60 * fs)
lfp_excerpt = raw_1d

print(f"Extracted lfp_excerpt shape: {lfp_excerpt.shape}, dtype: {lfp_excerpt.dtype}")
print(f"Mean: {lfp_excerpt.mean():.4f}, Std: {lfp_excerpt.std():.4f}")

Extracted lfp_excerpt shape: (2442112,), dtype: float64
Mean: 0.0000, Std: 0.0001


In [35]:
import sys
from pathlib import Path

# Add the parent directory (where 'src' lives) to sys.path
root_dir = Path.cwd().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# Verify it found src
print("Looking in:", root_dir)
print("Contains src:", (root_dir / "src").exists())

from src.mapping.smooth_env import DEFAULT_FREQ_BANDS, smooth_env

# 1. Run smooth_env on your short slice
# Note: DEFAULT_FREQ_BANDS is imported directly from your port
env = smooth_env(lfp_excerpt, fs, DEFAULT_FREQ_BANDS)

# 2. Trim boundary NaNs (matching the script's prepare() logic)
valid = np.isfinite(env).all(axis=1)
env_clean = env[valid]

print(f"Envelope shape after trimming: {env_clean.shape}")
print(f"Expected shape: (N_valid_samples, {len(DEFAULT_FREQ_BANDS)})")

Looking in: /Users/Sam/dc-allen-data-acquire
Contains src: True
Envelope shape after trimming: (2437112, 10)
Expected shape: (N_valid_samples, 10)


In [36]:
from src.mapping.get_motifs import get_motifs

# Prepare input matching the script: list of (n_bands, n_samples)
envelopes_input = [env_clean.T]

# Run get_motifs on this test excerpt
# Keep n_null small (e.g., 5) for quick unit-testing
result = get_motifs(envelopes_input, fs=fs, n_null=20, seed=0, verbose=True)

print(f"\nDiscovered motif pairs: {result.n_motifs}")
if result.n_motifs > 0:
    print(f"Motifs matrix shape: {result.motifs.shape}")
    print("Discovered motif weights:\n", np.round(result.motifs, 3))

  [real data] 9202 timepoints, candidates per threshold: [28, 49, 82, 164, 456] (9.2s)
  [null 1/20] 9572 timepoints, candidates per threshold: [33, 59, 97, 217, 656] (7.9s)
  [null 2/20] 9607 timepoints, candidates per threshold: [34, 64, 109, 204, 654] (7.0s)
  [null 3/20] 9593 timepoints, candidates per threshold: [32, 56, 96, 216, 627] (6.8s)
  [null 4/20] 9586 timepoints, candidates per threshold: [35, 58, 96, 215, 632] (6.5s)
  [null 5/20] 9585 timepoints, candidates per threshold: [43, 63, 119, 219, 634] (7.0s)
  [null 6/20] 9612 timepoints, candidates per threshold: [37, 59, 111, 227, 637] (6.6s)
  [null 7/20] 9618 timepoints, candidates per threshold: [32, 59, 105, 228, 649] (6.4s)
  [null 8/20] 9579 timepoints, candidates per threshold: [36, 68, 108, 246, 654] (6.5s)
  [null 9/20] 9594 timepoints, candidates per threshold: [38, 48, 98, 232, 648] (6.7s)
  [null 10/20] 9588 timepoints, candidates per threshold: [32, 50, 90, 214, 637] (7.9s)
  [null 11/20] 9621 timepoints, candi

In [41]:
import inspect
from src.mapping.smooth_env import DEFAULT_FREQ_BANDS, smooth_env

print("DEFAULT_FREQ_BANDS:\n", DEFAULT_FREQ_BANDS)
print("\nsmooth_env signature and docstring:")
print(inspect.getsource(smooth_env))

DEFAULT_FREQ_BANDS:
 [[ 1.  4.]
 [ 3.  6.]
 [ 5.  9.]
 [ 8. 12.]
 [10. 15.]
 [13. 20.]
 [18. 30.]
 [25. 40.]
 [35. 55.]
 [50. 80.]]

smooth_env signature and docstring:
def smooth_env(
    lfp: np.ndarray,
    fs: float,
    freq_bands: np.ndarray = DEFAULT_FREQ_BANDS,
    smooth_sec: float = 5.0,
    edge_sec: float = 10.0,
) -> np.ndarray:
    """Port of SmoothEnv(LFP, LFPfs, freqBands).

    Returns a (time, bands) array: each band's Hilbert envelope, smoothed with a
    gausswin(round(5 * fs), 1) kernel applied forward and backward, the first and
    last `edge_sec` seconds set to NaN, then z-scored over time (ignoring NaNs).
    """
    x = np.asarray(lfp, dtype=float).ravel()
    freq_bands = np.atleast_2d(np.asarray(freq_bands, dtype=float))

    filt = np.empty((x.size, len(freq_bands)))
    for f, (low, high) in enumerate(freq_bands):
        filt[:, f] = signal.sosfiltfilt(butterfilter(fs, low, high), x)
    env = np.abs(signal.hilbert(filt, axis=0))

    w = gausswin(mround(

In [6]:
### THE GOOD ONE
import sys
from pathlib import Path

# Add the parent directory (where 'src' lives) to sys.path
root_dir = Path.cwd().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# Verify it found src
print("Looking in:", root_dir)
print("Contains src:", (root_dir / "src").exists())

import numpy as np
from src.mapping.smooth_env import smooth_env
from src.mapping.get_motifs import get_motifs

fs = 250.0 
mishler_6_bands = np.array([
    [1.0, 4.0],
    [4.0, 8.0],
    [8.0, 12.0],
    [12.0, 30.0],
    [30.0, 60.0],
    [60.0, 100.0]
])

# Use 5 minutes (300 seconds = 75,000 samples) instead of 16 minutes
n_samples = int(300 * fs)
all_envelopes = []

for ch in range(lfp.shape[0]):
    raw_ch = lfp[ch, 0]
    while isinstance(raw_ch, np.ndarray) and raw_ch.dtype == object:
        raw_ch = raw_ch[0]
    raw_1d = np.squeeze(raw_ch).astype(np.float64)[:n_samples]
    
    env = smooth_env(raw_1d, fs, mishler_6_bands)
    valid = np.isfinite(env).all(axis=1)
    all_envelopes.append(env[valid].T)

print(f"Prepared {len(all_envelopes)} channels, {all_envelopes[0].shape[1]} samples each.")

# Run with n_null=3 first to verify it completes quickly (< 20 seconds)
result = get_motifs(all_envelopes, fs=fs, n_null=3, seed=0, verbose=True)
print(f"Done! Discovered motifs: {result.n_motifs}")

Looking in: /Users/Sam/dc-allen-data-acquire
Contains src: True
Prepared 6 channels, 70000 samples each.
  [real data] 1465 timepoints, candidates per threshold: [12, 16, 31, 48, 61] (0.2s)
  [null 1/3] 1578 timepoints, candidates per threshold: [13, 17, 27, 62, 62] (0.2s)
  [null 2/3] 1600 timepoints, candidates per threshold: [11, 23, 32, 56, 62] (0.2s)
  [null 3/3] 1589 timepoints, candidates per threshold: [12, 17, 27, 59, 62] (0.2s)
Done! Discovered motifs: 2


In [9]:
import numpy as np
import pandas as pd

port_motifs = result.motifs  # Shape: (6, 2 * n_pairs)
n_pairs = result.n_motifs
n_ref = finalMotifs.shape[0]

# Proper Mishler indexing:
# First n_pairs columns are Positive poles, next n_pairs are Negative poles
positive_poles = port_motifs[:, :n_pairs]
negative_poles = port_motifs[:, n_pairs:]

print(f"Testing {n_pairs} motif pairs against all {n_ref} reference rows...")

best_matches = []

for m_idx in range(n_pairs):
  m_pos = positive_poles[:, m_idx]

  corrs = []
  for r_idx in range(n_ref):
    ref_vec = finalMotifs[r_idx, :]
    r = np.corrcoef(m_pos, ref_vec)[0, 1]
    corrs.append(r)

  corrs = np.array(corrs)

  # Find best matching reference row (either aligned or inverted)
  best_pos_idx = int(np.nanargmax(corrs))
  best_pos_r = corrs[best_pos_idx]

  best_neg_idx = int(np.nanargmin(corrs))
  best_neg_r = corrs[best_neg_idx]

  # Pick whichever direction has higher magnitude
  if abs(best_pos_r) >= abs(best_neg_r):
    best_row = best_pos_idx
    best_r = best_pos_r
    aligned_polarity = "+ (direct)"
  else:
    best_row = best_neg_idx
    best_r = -best_neg_r  # inverting sign
    aligned_polarity = "- (inverted sign)"

  best_matches.append({
      "Port Motif": f"Motif {m_idx + 1}",
      "Best Ref Row": best_row,
      "Polarity Match": aligned_polarity,
      "Max Pearson |r|": round(abs(best_r), 4),
      "Signed r": round(best_r, 4),
      "Status": "PASS" if abs(best_r) >= 0.75 else "INSPECT",
  })

summary_df = pd.DataFrame(best_matches)
display(summary_df)

Testing 2 motif pairs against all 76 reference rows...


,Port Motif,Best Ref Row,Polarity Match,Max Pearson |r|,Signed r,Status
0,Motif 1,0,+ (direct),0.3364,0.3364,INSPECT
1,Motif 2,58,- (inverted sign),0.7953,0.7953,PASS


In [8]:
band_names = ["Delta (1-4)", "Theta (4-8)", "Alpha (8-12)", "Beta (12-30)", "Low-Gamma (30-60)", "High-Gamma (60-100)"]

for row in matches:
    pole_idx = int(row["Discovered Pole"].split()[1]) - 1
    ref_idx = row["Best Matched Ref Row"]
    
    print(f"\n==========================================")
    print(f"{row['Discovered Pole']} vs. Reference finalMotifs[{ref_idx}] (r = {row['Pearson r']})")
    print(f"==========================================")
    
    comp_table = pd.DataFrame({
        "Band": band_names,
        "Port Value": np.round(port_motifs[:, pole_idx], 3),
        "Jonathan Ref": np.round(finalMotifs[ref_idx, :], 3),
        "Absolute Diff": np.round(np.abs(port_motifs[:, pole_idx] - finalMotifs[ref_idx, :]), 3)
    })
    print(comp_table.to_string(index=False))


Pole 1 (M1+) vs. Reference finalMotifs[0] (r = 0.3364)
               Band  Port Value  Jonathan Ref  Absolute Diff
        Delta (1-4)      -0.549        -0.670          0.121
        Theta (4-8)      -0.029         0.670          0.699
       Alpha (8-12)       0.212        -0.358          0.571
       Beta (12-30)       0.559         0.358          0.201
  Low-Gamma (30-60)       0.900        -0.182          1.082
High-Gamma (60-100)       0.888         0.182          0.705

Pole 2 (M1-) vs. Reference finalMotifs[10] (r = 0.1598)
               Band  Port Value  Jonathan Ref  Absolute Diff
        Delta (1-4)       1.000        -0.037          1.037
        Theta (4-8)      -0.249         0.037          0.287
       Alpha (8-12)      -0.806        -0.233          0.573
       Beta (12-30)       0.274         0.233          0.041
  Low-Gamma (30-60)       0.534         0.430          0.104
High-Gamma (60-100)       0.633        -0.430          1.063

Pole 3 (M2+) vs. Reference final

In [10]:
import numpy as np

print("=== Keys in reference MAT file ===")
keys = [k for k in fGLFP_data.keys() if not k.startswith("__")]
for k in keys:
    val = fGLFP_data[k]
    print(f"Key: {k:<20} Shape: {getattr(val, 'shape', 'scalar'):<15} Type: {type(val)}")

# Check if there are channel maps, region names, or session IDs
for probe_key in ["channel", "channels", "region", "regions", "area", "areas", "probe", "probes", "mouse", "mice", "session", "sessions", "info", "meta"]:
    if probe_key in fGLFP_data:
        print(f"\nMetadata '{probe_key}':\n", fGLFP_data[probe_key])

=== Keys in reference MAT file ===


TypeError: unsupported format string passed to tuple.__format__

In [11]:
# What are the 6 rows of lfp? Channels on one probe, or 6 different sessions/mice?
print("lfp shape:", lfp.shape)
for r in range(lfp.shape[0]):
    for c in range(lfp.shape[1]):
        item = lfp[r, c]
        while isinstance(item, np.ndarray) and item.dtype == object:
            item = item[0]
        print(f"Row {r}, Col {c}: length = {len(item.squeeze())}")

lfp shape: (6, 3)
Row 0, Col 0: length = 2442112
Row 0, Col 1: length = 2442752
Row 0, Col 2: length = 2472448
Row 1, Col 0: length = 2443520
Row 1, Col 1: length = 2442240
Row 1, Col 2: length = 2454144
Row 2, Col 0: length = 2043392
Row 2, Col 1: length = 2086912
Row 2, Col 2: length = 1935488
Row 3, Col 0: length = 2498176
Row 3, Col 1: length = 2452864
Row 3, Col 2: length = 2445696
Row 4, Col 0: length = 2494720
Row 4, Col 1: length = 3565184
Row 4, Col 2: length = 2547200
Row 5, Col 0: length = 2714496
Row 5, Col 1: length = 2547456
Row 5, Col 2: length = 2483968


In [12]:
# Inspect keys and metadata in Jonathan's reference file
mat_keys = [k for k in fGLFP_data.keys() if not k.startswith("__")]
print("Reference MAT keys:", mat_keys)

for k in mat_keys:
    val = fGLFP_data[k]
    shape = getattr(val, "shape", "scalar")
    dtype = getattr(val, "dtype", type(val))
    print(f"  {k:<20} | shape: {str(shape):<15} | dtype: {dtype}")

Reference MAT keys: ['finalMotifs', 'shifts', 'finalMotifsAll']
  finalMotifs          | shape: (76, 6)         | dtype: float64
  shifts               | shape: (76, 21)        | dtype: int32
  finalMotifsAll       | shape: (21, 5)         | dtype: object


In [13]:
### THE GOOD ONE
import sys
from pathlib import Path

# Add the parent directory (where 'src' lives) to sys.path
root_dir = Path.cwd().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import numpy as np
import inspect
from src.mapping.smooth_env import smooth_env

# 1. Inspect smooth_env source to check downsampling and filter design
print(inspect.getsource(smooth_env))

def smooth_env(
    lfp: np.ndarray,
    fs: float,
    freq_bands: np.ndarray = DEFAULT_FREQ_BANDS,
    smooth_sec: float = 5.0,
    edge_sec: float = 10.0,
) -> np.ndarray:
    """Port of SmoothEnv(LFP, LFPfs, freqBands).

    Returns a (time, bands) array: each band's Hilbert envelope, smoothed with a
    gausswin(round(5 * fs), 1) kernel applied forward and backward, the first and
    last `edge_sec` seconds set to NaN, then z-scored over time (ignoring NaNs).
    """
    x = np.asarray(lfp, dtype=float).ravel()
    freq_bands = np.atleast_2d(np.asarray(freq_bands, dtype=float))

    filt = np.empty((x.size, len(freq_bands)))
    for f, (low, high) in enumerate(freq_bands):
        filt[:, f] = signal.sosfiltfilt(butterfilter(fs, low, high), x)
    env = np.abs(signal.hilbert(filt, axis=0))

    w = gausswin(mround(smooth_sec * fs), 1)
    w /= w.sum()
    smooth = fir_filtfilt(w, env)

    edge = mround(edge_sec * fs)
    smooth[:edge] = np.nan
    smooth[smooth.shape[0] - edge:] 

In [14]:
import numpy as np
import pandas as pd
from src.mapping.smooth_env import butterfilter, gausswin, fir_filtfilt, mround
from scipy import signal

# Canonical 6 bands for Mishler replication
bands_6 = np.array([
    [1.0, 4.0],
    [4.0, 8.0],
    [8.0, 12.0],
    [12.0, 30.0],
    [30.0, 60.0],
    [60.0, 100.0]
], dtype=float)

# 1. Extract 30-second test excerpt from session 0, awake (col 0)
fs = 250.0  # Assumes pre-decimated 250 Hz
raw_ch = lfp[0, 0]
while isinstance(raw_ch, np.ndarray) and raw_ch.dtype == object:
    raw_ch = raw_ch[0]
raw_trace = np.squeeze(raw_ch).astype(np.float64)[:int(30 * fs)]

# 2. Stage 1 & 2: Bandpass + Hilbert
filt = np.empty((raw_trace.size, len(bands_6)))
env = np.empty((raw_trace.size, len(bands_6)))
for f, (low, high) in enumerate(bands_6):
    sos = butterfilter(fs, low, high, order=6)
    filt[:, f] = signal.sosfiltfilt(sos, raw_trace)
    env[:, f] = np.abs(signal.hilbert(filt[:, f]))

# 3. Stage 3: Gaussian Smoothing
w = gausswin(mround(5.0 * fs), alpha=1.0)
w /= w.sum()
smooth = fir_filtfilt(w, env)

# 4. Stage 4: NaN Trimming & Z-scoring
edge = mround(10.0 * fs)
smooth_trimmed = smooth.copy()
smooth_trimmed[:edge] = np.nan
smooth_trimmed[-edge:] = np.nan

mu = np.nanmean(smooth_trimmed, axis=0)
sigma = np.nanstd(smooth_trimmed, axis=0, ddof=1)
z_env = (smooth_trimmed - mu) / sigma

# Summary metrics across the 6 bands
band_labels = ["Delta (1-4)", "Theta (4-8)", "Alpha (8-12)", "Beta (12-30)", "Low-Gamma (30-60)", "High-Gamma (60-100)"]
stats = []
for i, name in enumerate(band_labels):
    valid_z = z_env[:, i][np.isfinite(z_env[:, i])]
    stats.append({
        "Band": name,
        "Raw Env Mean (uV)": round(np.mean(env[:, i]), 3),
        "Raw Env Std (uV)": round(np.std(env[:, i]), 3),
        "Smooth Mean (uV)": round(np.nanmean(smooth[:, i]), 3),
        "Z-score Mean": round(np.mean(valid_z), 6),
        "Z-score Std": round(np.std(valid_z, ddof=1), 6)
    })

df_stage_summary = pd.DataFrame(stats)
display(df_stage_summary)

,Band,Raw Env Mean (uV),Raw Env Std (uV),Smooth Mean (uV),Z-score Mean,Z-score Std
0,Delta (1-4),0.0,0.0,0.0,-0.0,1.0
1,Theta (4-8),0.0,0.0,0.0,0.0,1.0
2,Alpha (8-12),0.0,0.0,0.0,-0.0,1.0
3,Beta (12-30),0.0,0.0,0.0,0.0,1.0
4,Low-Gamma (30-60),0.0,0.0,0.0,0.0,1.0
5,High-Gamma (60-100),0.0,0.0,0.0,0.0,1.0


In [15]:
import numpy as np
import pandas as pd
from src.mapping.smooth_env import smooth_env

# 1. Slice 30 seconds at 250 Hz from session 0, awake
fs = 250.0
raw_ch = lfp[0, 0]
while isinstance(raw_ch, np.ndarray) and raw_ch.dtype == object:
  raw_ch = raw_ch[0]
raw_trace = np.squeeze(raw_ch).astype(np.float64)[: int(30 * fs)]

# 2. Run Python port
smooth_py = smooth_env(raw_trace, fs, bands_6)

# 3. If you have Jonathan's butterfilter.m or MATLAB running, save smoothLFPNorm
# from MATLAB to smooth_matlab.mat and compare:
# smooth_mat = sio.loadmat('smooth_matlab.mat')['smoothLFPNorm']

In [16]:
import numpy as np
from scipy import signal
from src.mapping.smooth_env import butterfilter

fs = 250.0
low, high = 4.0, 8.0  # Theta band example

# Generate Python SOS matrix
sos_py = butterfilter(fs, low, high, order=6)

print(f"SOS Matrix Shape: {sos_py.shape} (Expected: (3, 6) for 3 second-order sections)")

# Compute frequency response
w, h = signal.sosfreqz(sos_py, worN=2048, fs=fs)
passband_gain = np.max(np.abs(h))
f_peak = w[np.argmax(np.abs(h))]

print(f"Peak Frequency: {f_peak:.2f} Hz (Centered between {low} and {high} Hz)")
print(f"Passband Peak Gain: {passband_gain:.6f} (Unit gain = 1.000000)")

# Assert passband unity gain within float precision
assert np.isclose(passband_gain, 1.0, atol=1e-5), "Filter gain scaling mismatch!"
print("Filter architecture and gain scaling verified.")

SOS Matrix Shape: (3, 6) (Expected: (3, 6) for 3 second-order sections)
Peak Frequency: 5.68 Hz (Centered between 4.0 and 8.0 Hz)
Passband Peak Gain: 1.000000 (Unit gain = 1.000000)
Filter architecture and gain scaling verified.


In [17]:
import inspect
from src.mapping.get_motifs import get_motifs

# Print the top-level functions and docstring of get_motifs
print(inspect.getsource(get_motifs))

def get_motifs(
    lfp_in,
    fs: float,
    corr_thresholds=CORR_THRESHOLDS,
    n_null: int = 20,
    seed=None,
    verbose: bool = False,
) -> MotifResult:
    """Port of getMotifs(LFPIn, LFPfs).

    lfp_in: list of (bands, time) arrays, one per session, same bands in the same
    order. Must not contain NaN. fs: sampling rate of the time axis in Hz.
    """
    sessions = [np.asarray(s, dtype=float) for s in lfp_in]
    n_bands = sessions[0].shape[0]
    if any(s.ndim != 2 or s.shape[0] != n_bands for s in sessions):
        raise ValueError("every session must be a (bands, time) array with the same number of bands")
    if not all(np.isfinite(s).all() for s in sessions):
        raise ValueError("input contains NaN/inf; trim SmoothEnv's NaN edges before calling get_motifs")

    hop = mround(fs)
    session_of = np.concatenate([np.full(s.shape[1], i) for i, s in enumerate(sessions)])[::hop]
    shifts = _null_shifts(n_bands, fs, n_null, np.random.default_rng(seed))
    thresho

In [18]:
import numpy as np
import pandas as pd
from scipy import signal
from src.mapping.smooth_env import smooth_env

# 1. Setup 6 canonical Mishler bands
bands_6 = np.array([
    [1.0, 4.0],
    [4.0, 8.0],
    [8.0, 12.0],
    [12.0, 30.0],
    [30.0, 60.0],
    [60.0, 100.0]
], dtype=float)

fs = 250.0  # sampling rate (Hz)
band_names = ["Delta (1-4)", "Theta (4-8)", "Alpha (8-12)", "Beta (12-30)", "Low-Gamma (30-60)", "High-Gamma (60-100)"]

# 2. Extract a 60-second test excerpt from session 0, awake
raw_entry = lfp[0, 0]
while isinstance(raw_entry, np.ndarray) and raw_entry.dtype == object:
    raw_entry = raw_entry[0]
raw_1d = np.squeeze(raw_entry).astype(np.float64)[:int(60 * fs)]

# 3. Stage 1: Envelopes & Stage 2: Z-scoring
z_env = smooth_env(raw_1d, fs=fs, freq_bands=bands_6)
valid_mask = np.isfinite(z_env).all(axis=1)
z_valid = z_env[valid_mask]  # Shape: (N_valid, 6)

# 4. Stage 3: BCM (Binary Concordance Measure)
# Downsample to 1 Hz (hop = round(fs)) matching Mishler's 1 Hz bins
hop = int(round(fs))
env_1hz = z_valid[::hop]

# Binary signs across the 6 frequency bands
signs_1hz = np.sign(env_1hz)  # Matrix of {-1, +1}

# Filter out trivial broadband shifts (all bands having the same sign)
nontrivial_mask = np.abs(signs_1hz.sum(axis=1)) != signs_1hz.shape[1]
signs_nontrivial = signs_1hz[nontrivial_mask]

# Compute 6x6 pairwise Binary Concordance Matrix: E[sign(f_i) * sign(f_j)]
# Mean product of signs ranges from -1.0 (pure opponent) to +1.0 (pure co-activation)
bcm_matrix = (signs_nontrivial.T @ signs_nontrivial) / signs_nontrivial.shape[0]

print(f"Valid 1 Hz timepoints: {signs_1hz.shape[0]}")
print(f"Nontrivial timepoints: {signs_nontrivial.shape[0]} ({signs_nontrivial.shape[0]/signs_1hz.shape[0]:.1%})")
print("\n6x6 Binary Concordance Matrix (BCM):")
bcm_df = pd.DataFrame(np.round(bcm_matrix, 3), index=band_names, columns=band_names)
display(bcm_df)

Valid 1 Hz timepoints: 40
Nontrivial timepoints: 35 (87.5%)

6x6 Binary Concordance Matrix (BCM):


,Delta (1-4),Theta (4-8),Alpha (8-12),Beta (12-30),Low-Gamma (30-60),High-Gamma (60-100)
Delta (1-4),1.000,0.200,-0.143,-0.257,-0.200,-0.371
Theta (4-8),0.200,1.000,-0.029,-0.143,0.029,-0.257
Alpha (8-12),-0.143,-0.029,1.000,0.200,-0.200,0.086
Beta (12-30),-0.257,-0.143,0.200,1.000,0.486,0.657
Low-Gamma (30-60),-0.200,0.029,-0.200,0.486,1.000,0.486
High-Gamma (60-100),-0.371,-0.257,0.086,0.657,0.486,1.000


In [19]:
comparison_metrics = []

for i, name in enumerate(band_names):
    col = z_valid[:, i]
    # Check normalization
    mean_val = float(np.mean(col))
    std_val = float(np.std(col, ddof=1))
    
    # Check binary distribution: fraction of time positive vs negative
    frac_pos = float(np.mean(signs_nontrivial[:, i] > 0))
    
    # Average concordance of this band with all other bands
    mean_concordance = float(np.mean(np.delete(bcm_matrix[i, :], i)))
    
    comparison_metrics.append({
        "Frequency Band": name,
        "Z-score Mean": f"{mean_val:+.4e}",
        "Z-score Std": f"{std_val:.4f}",
        "Positive Fraction": f"{frac_pos:.2%}",
        "Mean Cross-Band BCM": f"{mean_concordance:+.3f}",
        "Status": "PASS" if (abs(mean_val) < 1e-5 and abs(std_val - 1.0) < 1e-4) else "FAIL"
    })

deliverable_table = pd.DataFrame(comparison_metrics)
display(deliverable_table)

,Frequency Band,Z-score Mean,Z-score Std,Positive Fraction,Mean Cross-Band BCM,Status
0,Delta (1-4),+1.7235e-14,1.0000,40.00%,-0.154,PASS
1,Theta (4-8),-1.5780e-14,1.0000,22.86%,-0.040,PASS
2,Alpha (8-12),-2.9763e-14,1.0000,40.00%,-0.017,PASS
3,Beta (12-30),+2.5057e-14,1.0000,45.71%,+0.189,PASS
4,Low-Gamma (30-60),+8.0945e-15,1.0000,37.14%,+0.120,PASS
5,High-Gamma (60-100),+1.6871e-14,1.0000,45.71%,+0.120,PASS


In [20]:
import numpy as np
import pandas as pd
from src.mapping.smooth_env import smooth_env

# 1. Jonathan's 6 canonical frequency bands
bands_6 = np.array([
    [1.0, 4.0],
    [4.0, 8.0],
    [8.0, 12.0],
    [12.0, 30.0],
    [30.0, 60.0],
    [60.0, 100.0]
], dtype=float)

fs = 250.0  # sampling rate
duration_sec = 60
n_samples = int(duration_sec * fs)

# 2. Extract 60-second slice from session 0, awake (col 0)
raw_ch = lfp[0, 0]
while isinstance(raw_ch, np.ndarray) and raw_ch.dtype == object:
    raw_ch = raw_ch[0]
raw_1d = np.squeeze(raw_ch).astype(np.float64)[:n_samples]

# 3. Stage 1 & 2: Envelopes + Z-scoring via Python port
z_env = smooth_env(raw_1d, fs, bands_6)
valid_mask = np.isfinite(z_env).all(axis=1)
z_valid = z_env[valid_mask]  # Shape: (N_valid, 6)

# 4. Stage 3: Downsample to 1 Hz & calculate Binary Concordance (BCM) signs
hop = int(round(fs))
env_1hz = z_valid[::hop]
signs_1hz = np.sign(env_1hz)

# Remove trivial all-same-sign points
nontrivial = np.abs(signs_1hz.sum(axis=1)) != signs_1hz.shape[1]
signs_nontrivial = signs_1hz[nontrivial]

print(f"Total 1 Hz timepoints: {len(signs_1hz)}")
print(f"Nontrivial timepoints: {len(signs_nontrivial)}")

Total 1 Hz timepoints: 40
Nontrivial timepoints: 35


In [21]:
# Project 1 Hz sign states onto Jonathan's 76 reference motifs
# signs_nontrivial is (T, 6), finalMotifs is (76, 6) -> projections is (T, 76)
projections = signs_nontrivial @ finalMotifs.T

# 1. Measure mean activation / alignment across time for each reference motif
mean_projection = np.mean(projections, axis=0)

# 2. Measure pole occupancy: which reference motif has the strongest alignment at each second
dominant_motif_per_sec = np.argmax(np.abs(projections), axis=1)
unique_motifs, counts = np.unique(dominant_motif_per_sec, return_counts=True)

# 3. Summary of top active reference motifs during this 60-second excerpt
top_indices = np.argsort(-np.abs(mean_projection))[:5]

band_names = ["Delta", "Theta", "Alpha", "Beta", "Low-Gamma", "High-Gamma"]
print("=== Top 5 Most Active Reference Motifs in this 60s Excerpt ===")
for rank, idx in enumerate(top_indices):
    ref_vector = finalMotifs[idx, :]
    print(f"\nRank {rank+1}: Reference Motif Row {idx}")
    print(f"  Mean Projection Score: {mean_projection[idx]:.3f}")
    print(f"  Dominant seconds: {np.sum(dominant_motif_per_sec == idx)} / {len(signs_nontrivial)} s")
    print(f"  Spectral weights: {dict(zip(band_names, np.round(ref_vector, 3)))}")

=== Top 5 Most Active Reference Motifs in this 60s Excerpt ===

Rank 1: Reference Motif Row 54
  Mean Projection Score: -0.274
  Dominant seconds: 0 / 35 s
  Spectral weights: {'Delta': np.float64(-0.797), 'Theta': np.float64(0.797), 'Alpha': np.float64(0.207), 'Beta': np.float64(-0.207), 'Low-Gamma': np.float64(-0.135), 'High-Gamma': np.float64(0.135)}

Rank 2: Reference Motif Row 52
  Mean Projection Score: -0.272
  Dominant seconds: 0 / 35 s
  Spectral weights: {'Delta': np.float64(-0.85), 'Theta': np.float64(0.85), 'Alpha': np.float64(0.064), 'Beta': np.float64(-0.064), 'Low-Gamma': np.float64(-0.157), 'High-Gamma': np.float64(0.157)}

Rank 3: Reference Motif Row 53
  Mean Projection Score: -0.271
  Dominant seconds: 0 / 35 s
  Spectral weights: {'Delta': np.float64(-0.823), 'Theta': np.float64(0.823), 'Alpha': np.float64(0.124), 'Beta': np.float64(-0.124), 'Low-Gamma': np.float64(-0.145), 'High-Gamma': np.float64(0.145)}

Rank 4: Reference Motif Row 51
  Mean Projection Score: -0.

In [22]:
import scipy.io as sio

# Save the exact 60s snippet to test in MATLAB
sio.savemat("test_excerpt_60s.mat", {
    "raw_excerpt": raw_1d,
    "fs": fs,
    "bands_6": bands_6,
    "port_z_env": z_env
})
print("Saved test_excerpt_60s.mat")

Saved test_excerpt_60s.mat


In [36]:
from pathlib import Path
import scipy.io as sio

mishler_dir = Path("/Users/Sam/dc-allen-data-acquire/data/mishler_results")

print("Files in mishler_results:")
for p in mishler_dir.glob("*"):
    print(f"  {p.name} ({p.stat().st_size / 1e6:.1f} MB)")

Files in mishler_results:
  finalGlutLFPMotifs.mat (11.4 MB)
  LFP Data.mat (165.7 MB)


In [25]:
fGLFP_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Sun Jul 26 11:58:54 2026',
 '__version__': '1.0',
 '__globals__': [],
 'finalMotifs': array([[-6.70087464e-01,  6.70087464e-01, -3.58255452e-01,
          3.58255452e-01, -1.82496075e-01,  1.82496075e-01],
        [-6.62274052e-01,  6.62274052e-01, -2.93563995e-01,
          2.93563995e-01, -2.15070644e-01,  2.15070644e-01],
        [-3.78425656e-01,  3.78425656e-01, -1.07045801e-01,
          1.07045801e-01, -1.13029827e-01,  1.13029827e-01],
        [ 5.03790087e-02, -5.03790087e-02,  1.65639292e-01,
         -1.65639292e-01,  7.22135008e-02, -7.22135008e-02],
        [ 3.96967930e-01, -3.96967930e-01,  4.41108239e-01,
         -4.41108239e-01,  2.01726845e-01, -2.01726845e-01],
        [ 4.88979592e-01, -4.88979592e-01,  5.58560350e-01,
         -5.58560350e-01,  2.36459969e-01, -2.36459969e-01],
        [ 4.03965015e-01, -4.03965015e-01,  5.45436468e-01,
         -5.45436468e-01,  2.47841444e-01, -2.47841444e-01],

In [38]:
import numpy as np
import pandas as pd

band_names = ["Delta (1-4)", "Theta (4-8)", "Alpha (8-12)", "Beta (12-30)", "Low-Gamma (30-60)", "High-Gamma (60-100)"]

# Load discovered motifs and reference
discovered = np.load("/Users/Sam/dc-allen-data-acquire/results_task4/discovered_motifs.npy")

df_inspect = pd.DataFrame({
    "Band": band_names,
    "Discovered M2+": np.round(discovered[:, 1], 3),
    "Ref Row 57": np.round(finalMotifs[57, :], 3),
    "Discovered M1+": np.round(discovered[:, 0], 3),
    "Ref Row 17": np.round(finalMotifs[17, :], 3)
})

print(df_inspect.to_string(index=False))

               Band  Discovered M2+  Ref Row 57  Discovered M1+  Ref Row 17
        Delta (1-4)          -0.887      -0.560          -0.183      -0.759
        Theta (4-8)           0.889       0.560          -0.077       0.759
       Alpha (8-12)           0.482       0.385           0.191      -0.740
       Beta (12-30)          -0.519      -0.385           0.419       0.740
  Low-Gamma (30-60)          -0.107      -0.035           0.943       0.442
High-Gamma (60-100)           0.043       0.035           0.896      -0.442
